# 04: Contar, sumar y promediar: funciones de agregación

COUNT, SUM, AVG, MIN y MAX — convertir miles de filas en un solo número que responde una pregunta de negocio.



Hasta ahora, tus consultas devuelven filas individuales: cada cliente, cada pedido, cada producto. 

- Cuando el director financiero pregunta "¿cuánto vendimos este trimestre?", no quiere ver 14.000 filas de pedidos: quiere UN número
- Cuando marketing pregunta "¿cuál es nuestro ticket medio?", espera una respuesta, no una tabla interminable. 

Las funciones de agregación son exactamente eso: toman N filas y las colapsan en un solo valor que responde una pregunta de negocio.


**La analogía perfecta:**

Imagina que tienes un montón de recibos de compra sobre la mesa. 

- "Contar" los recibos es COUNT. 
- "Sumar" todos los importes es SUM. 
- "Calcular la media" es AVG. 
- "Buscar el más alto" es MAX. 

Son operaciones que hacemos mentalmente con datos pequeños, pero que SQL ejecuta sobre millones de filas en milisegundos.


## Las 5 funciones de agregación fundamentales


- **COUNT(*):** cuenta el número TOTAL de filas (incluye filas con NULLs)
- **COUNT(columna):** cuenta filas donde esa columna NO es NULL
- **COUNT(DISTINCT columna):** cuenta valores ÚNICOS (sin repetir)
- **SUM(columna):** suma todos los valores numéricos de la columna
- **AVG(columna):** media aritmética (ignora NULLs automáticamente)
- **MAX(columna):** el valor más grande (funciona con números, fechas y texto)
- **MIN(columna):** el valor más pequeño

```sql
-- Cuántos clientes tenemos en total?
SELECT COUNT(*) AS total_clientes
FROM clientes;

-- Cuántos hemos facturado en total?
SELECT ROUND(SUM(importe), 2) AS ventas_totales
FROM pedidos;

-- Cuál es el importe medio de un pedido (ticket medio)?
SELECT ROUND(AVG(importe), 2) AS ticket_medio
FROM pedidos;

-- Cuál es el pedido más grande y el más pequeño?
SELECT
    MAX(importe) AS pedido_mayor,
    MIN(importe) AS pedido_menor
FROM pedidos;
```

Cuatro queries que colapsan TODAS las filas en un solo resultado:

- COUNT(*): cuenta filas. Resultado: un número (ej: 5000 clientes).
- ROUND(SUM(importe), 2): suma todos los importes, redondeada a 2 decimales para reportes limpios.
- ROUND(AVG(importe), 2): media aritmética, redondeada a 2 decimales para legibilidad.
- MAX/MIN: extremos de la columna.

**Nota:** 

- Cada query devuelve UNA sola fila. Las funciones de agregación "colapsan" miles de filas en un solo valor resumen. 
- Es la diferencia fundamental respecto a un SELECT normal que devuelve múltiples filas.

## COUNT(*) vs COUNT(columna) vs COUNT(DISTINCT)


Esta es una distinción que separa a los juniors de los mids. 

COUNT tiene tres variantes con comportamientos muy diferentes, y confundirlas genera bugs sutiles en reportes financieros:


```sql
-- COUNT(*): cuenta TODAS las filas, incluso las que tienen NULLS
SELECT COUNT(*) AS total_filas
FROM clientes;

-- COUNT(ciudad): cuenta filas donde ciudad NO es NULL
-- (habrá menos si hay clientes sin ciudad)
SELECT COUNT(ciudad) AS con_ciudad
FROM clientes;

-- COUNT(DISTINCT ciudad): cuántas ciudades diferentes tenemos?
SELECT COUNT(DISTINCT ciudad) AS ciudades_unicas
FROM clientes;

-- La diferencia en acción:
SELECT
    COUNT(*) AS filas_totales
    COUNT(ciudad) AS con_ciudad,
    COUNT(DISTINCT ciudad) AS ciudades_distintas
FROM clientes;
```

Tres COUNTs, tres significados: elige el correcto

Diferencias críticas entre las tres variantes de COUNT:

1. **COUNT(*):** 
   - Cuenta TODAS las filas sin importar si tienen NULLs. 
   - Es el total absoluto.
2. **COUNT(ciudad):** 
   - Cuenta solo filas donde ciudad NO es NULL. 
   - Si 50 clientes no tienen ciudad, dará 50 menos que COUNT(*).
3. **COUNT(DISTINCT ciudad):** 
   - Cuenta valores ÚNICOS de ciudad 
   - Aquí, 14 ciudades diferentes, sin importar cuántos clientes hay en cada una 
   - No cuenta el «no consta», porque NULL no es un valor.

La última query combina los tres para ver la diferencia en acción. 

Es el patrón de "auditoría de datos" que te permite detectar NULLs y cardinalidad de una columna.



**Consejo de senior:**

- En mi segundo trabajo, un reporte financiero daba 2% menos de ventas que el ERP. 
- Después de 3 horas de debug, descubrí que alguien usaba COUNT(importe) en vez de COUNT(*), y había pedidos con importe NULL (pagos pendientes). COUNT(columna) ignora NULLs silenciosamente. 

Siempre verifica cuál estás usando.


## Combinar múltiples agregaciones en una sola query


1. No necesitas ejecutar 5 queries separadas para obtener 5 métricas 
2. SQL permite combinar todas las funciones de agregación en un solo SELECT 
3. Este patrón es la base de cualquier dashboard:


```sql
-- Dashboard de métricas en UNA sola query
SELECT
    COUNT(*) AS total_pedidos,
    COUNT(DISTINCT cliente_id) AS clientes_activos,
    ROUND(SUM(importe), 2) AS facturacion_total,
    ROUND(AVG(importe), 2) AS ticket_medio
    MAX(importe) AS pedido_max,
    MIN(importe) AS pedido_min,
    MAX(fecha) AS ultimo_pedido
FROM pedidos
WHERE estado = 'completado';
```

Un dashboard completo en una sola consulta

- COUNT(*) → total de pedidos completados.
- COUNT(DISTINCT cliente_id) → clientes únicos que han comprado (un cliente con 5 pedidos cuenta 1).
- ROUND(SUM(importe), 2) → facturación total, redondeada a 2 decimales para reportes limpios.
- ROUND(AVG(importe), 2) → ticket medio redondeado a 2 decimales.
- MAX(importe) → pedido más grande.
- MIN(importe) → pedido más pequeño.
- MAX(fecha) → fecha del último pedido.

Todas las funciones se ejecutan sobre el mismo conjunto de filas (las que pasan el WHERE). 

El resultado es UNA sola fila con 7 columnas: un resumen ejecutivo en una query.



1. **SUM:** sube de tipo solo: 
    - si sumas una columna de enteros te devuelve un entero grande, así que no se desborda por mucho que sumes. 
    - Lo que sí desborda es forzar el resultado de vuelta a un entero normal con ::INTEGER, y eso no da un número raro, da un error y la consulta no corre.
    -  Así que no lo hagas. 
2. **AVG:** devuelve un decimal con muchas cifras. 
    - Para eso está ROUND: ROUND(AVG(importe), 2) te lo deja en dos decimales. 
    - En dinero se redondea para MOSTRAR, nunca para calcular
    - Redondea al final, en la última línea, no en medio.



```sql
-- SUM devuelve un número grande: usa ROUND para dejarlo legible
SELECT ROUND(SUM(importe), 2) AS total 
FROM pedidos;

-- AVG da muchos decimales: usa ROUND
SELECT ROUND(AVG(importe), 2) AS media
FROM pedidos;

-- Combina ambos para reportes limpios
SELECT
    ROUND(SUM(importe), 2) AS total_ventas,
    ROUND(AVG(importe), 2) AS ticket_medio,
    ROUND(SUM(importe) / COUNT(DISTINCT cliente_id), 2) AS gasto_medio_por_cliente
FROM pedidos
WHERE estado = 'completado';
```

- ROUND(SUM(importe), 2) → suma redondeada a 2 decimales. Útil en reportes donde los céntimos importan pero las milésimas no.
- ROUND(AVG(importe), 2) → redondea la media a 2 decimales. Sin ROUND, AVG puede dar 47.38294712...
- SUM(importe) / COUNT(DISTINCT cliente_id) → gasto medio POR CLIENTE (distinto de ticket medio por pedido). Un cliente con 5 pedidos divide su gasto total entre 1.

ROUND(valor, decimales) es la función estándar en DuckDB, PostgreSQL, MySQL y SQLite. Se usa al final, para mostrar — no en medio de un cálculo.



## MIN y MAX: no solo para números


MAX y MIN funcionan con cualquier tipo que se pueda ordenar: 
- números
- fechas
- incluso texto (orden alfabético). 

Esto los hace extremadamente útiles para encontrar rangos temporales:


```sql
-- Rango de fechas de los pedidos
SELECT 
    MIN(fecha) AS primer_pedido,
    MAX(fecha) AS ultimo_pedido,
    MAX(fecha) - MIN(fecha) AS dias_de_actividad
FROM pedidos;

-- Rango de precios por categoría
SELECT
    MIN(precio) AS precio_minimo
    MAX(precio) AS precio_maximo,
    MAX(precio) - MIN(precio) AS rango_precios
FROM productos;
```

- MIN(fecha) → la fecha más antigua (primer pedido del dataset).
- MAX(fecha) → la fecha más reciente (último pedido).
- MAX(fecha) - MIN(fecha) → restar dos DATE en DuckDB devuelve un BIGINT: el número de días, un entero pelado. (No un INTERVAL, que es lo que sale al restar dos TIMESTAMP.). Te dice el "span" temporal de tus datos.

El mismo patrón con precios: MIN/MAX/diferencia te da el rango de precios del catálogo. 

- **MAX - MIN es una medida de dispersión básica.**

Las funciones de agregación ignoran NULLs (excepto COUNT(*)). Si tienes 100 pedidos y 5 tienen importe NULL, AVG(importe) calcula la media de los 95 que SÍ tienen valor. 

Esto puede sesgar tus resultados sin que te des cuenta.



## Agregación con filtro: WHERE antes de agregar


Puedes combinar WHERE con funciones de agregación para calcular métricas sobre un subconjunto de datos. 

1. El WHERE se ejecuta PRIMERO (filtra filas) 
2. LUEGO la agregación actúa sobre las filas que pasaron el filtro:



```sql
-- Solo pedidos completados de 2024
SELECT
    COUNT(*) AS pedidos_2024,
    ROUND(SUM(importe), 2) AS total_2024,
    ROUND(AVG(importe), 2) AS ticket_medio_2024
FROM pedidos
WHERE estado = 'completado'
    AND fecha >= '2024-01-01';
```

El orden de ejecución interno de esta query:

1. FROM pedidos → toma la tabla.
2. WHERE estado = 'completado' AND fecha >= '2024-01-01' → filtra filas (solo completados de 2024 en adelante).
3. SELECT con COUNT, SUM, AVG → calcula las agregaciones SOLO sobre las filas que pasaron el filtro.

- El WHERE reduce el conjunto de datos ANTES de que las funciones de agregación actúen. 
- Por eso COUNT(*) no cuenta todos los pedidos del historial: solo los que cumplen las dos condiciones.



**UNION ALL:** 

A veces necesitas apilar los resultados de varias queries en una sola tabla. 

UNION ALL combina dos SELECTs verticalmente (requisito: mismo número de columnas y tipos compatibles). 

**Ejemplo:**

1. Un SELECT para completados + UNION ALL + un SELECT para cancelados: una tabla con ambos. 
2. UNION ALL mantiene duplicados (más rápido); UNION sin ALL los elimina.

